In [ ]:
from entropy_filter import execute_entropy_filter, Ticker, LOOKBACK_WINDOW, THRESHOLD, N_BINS, startDate, endDate
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np

# 1. 正確解包回傳值
Ticker, filter_df = execute_entropy_filter(Ticker, LOOKBACK_WINDOW, THRESHOLD, N_BINS)

# 2. 抓取大盤 VIX 數據 (只需抓一次，取 Close 欄位)
vix_raw = yf.download("^VIX", period="2y")['Close']
# 計算 252 天滾動百分位
vix_p = vix_raw.rolling(252).rank(pct=True)
# 取出最新一天的 VIX 百分位值，供所有個股共用
latest_vix_p = vix_p.iloc[-1].item() if not vix_p.empty else np.nan

# 3. 定義核心計算函數 (逐檔處理)
def get_indicators(passed_list):
    results = [] # 用來收集所有股票的最新數據
    
    for ticker in passed_list:
        # 1. 明確設定 auto_adjust=False 嘗試保留 Adj Close 欄位
        df = yf.download(ticker, period="2y", auto_adjust=False)
        
        # 確保有足夠的資料列數計算 MA200 (至少 200 天)
        if df.empty or len(df) < 200:
            continue
            
        # 2. 處理 yfinance 新版可能的 MultiIndex 欄位問題
        # 使用 get_level_values(0) 抽取第一層 (即 Price 名稱層) 會比 droplevel 更穩定
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 3. 欄位防呆機制：優先使用 Adj Close，若無則降級使用 Close
        if 'Adj Close' in df.columns:
            df['adj_price'] = df['Adj Close']
        elif 'Close' in df.columns:
            df['adj_price'] = df['Close']
        else:
            print(f"警告：{ticker} 缺少價格欄位，略過此檔。")
            continue
        
        # --- 計算技術指標 ---
        df['ma200'] = ta.sma(df['adj_price'], length=200)
        df['rsi'] = ta.rsi(df['adj_price'], length=14)

        # ADX
        adx_df = ta.adx(df['High'], df['Low'], df['adj_price'], length=14)
        df['adx'] = adx_df['ADX_14'] if adx_df is not None else np.nan

        # ATR 相關
        df['atr'] = ta.atr(df['High'], df['Low'], df['adj_price'], length=14)
        df['atr_60d_avg'] = df['atr'].rolling(window=60).mean()

        # --- 百分位指標 (Percentile) ---
        # BBW Percentile
        bbands = ta.bbands(df['adj_price'], length=20, std=2)
        if bbands is not None and not bbands.empty:
            # 動態比對並取得正確的欄位名稱
            bbu_col = [col for col in bbands.columns if 'BBU' in col][0]
            bbl_col = [col for col in bbands.columns if 'BBL' in col][0]
            bbm_col = [col for col in bbands.columns if 'BBM' in col][0]
            
            # 使用抓取到的正確名稱進行計算
            bbw = (bbands[bbu_col] - bbands[bbl_col]) / bbands[bbm_col]
            df['bbw_percentile'] = bbw.rolling(252).rank(pct=True)
        else:
            df['bbw_percentile'] = np.nan
                
        # HV Percentile
        log_ret = np.log(df['adj_price'] / df['adj_price'].shift(1))
        hv = log_ret.rolling(20).std() * np.sqrt(252)
        df['hv_percentile'] = hv.rolling(252).rank(pct=True)

        # --- 基本面 YoY ---
        tk = yf.Ticker(ticker)
        financials = tk.financials
        
        if financials is not None and 'Total Revenue' in financials.index:
            rev = financials.loc['Total Revenue']
            yoy = rev.pct_change(periods=-1) # 計算年增率
            yoy_now = yoy.iloc[0] if len(yoy) > 0 else np.nan
            yoy_t1 = yoy.iloc[1] if len(yoy) > 1 else np.nan
            yoy_t2 = yoy.iloc[2] if len(yoy) > 2 else np.nan
        else:
            yoy_now = yoy_t1 = yoy_t2 = np.nan

        # --- 整合該檔股票最後一天的所有數據 ---
        latest_data = {
            'ticker': ticker,
            'adj_price': round(df['adj_price'].iloc[-1], 2),
            'ma200': round(df['ma200'].iloc[-1], 2),
            'adx': round(df['adx'].iloc[-1], 2),
            'rsi': round(df['rsi'].iloc[-1], 2),
            'atr': round(df['atr'].iloc[-1], 2),
            'atr_60d_avg': round(df['atr_60d_avg'].iloc[-1], 2),
            'bbw_percentile': round(df['bbw_percentile'].iloc[-1], 4),
            'hv_percentile': round(df['hv_percentile'].iloc[-1], 4),
            'vix_percentile': round(latest_vix_p, 4), # 使用全域變數
            'yoy_now': round(yoy_now, 4) if pd.notna(yoy_now) else np.nan,
            'yoy_t1': round(yoy_t1, 4) if pd.notna(yoy_t1) else np.nan,
            'yoy_t2': round(yoy_t2, 4) if pd.notna(yoy_t2) else np.nan
        }
        
        # 將字典加入清單
        results.append(latest_data)
            
    # 將包含所有股票最新數據的清單轉為 DataFrame
    final_df = pd.DataFrame(results)
    if not final_df.empty:
        final_df = final_df.set_index('ticker')
        
    return final_df

# 4. 執行並顯示最終結果
final_report = get_indicators(Ticker)

# 如果在 Jupyter Notebook 中，建議使用 display 使排版更易讀
display(final_report)

In [ ]:
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np

#1. 讀取股票清單
Ticker = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

# 2. 抓取大盤 VIX 數據 (只需抓一次，取 Close 欄位)
vix_raw = yf.download("^VIX", period="2y", progress=False)['Close']
# 計算 252 天滾動百分位
vix_p = vix_raw.rolling(252).rank(pct=True)
# 確保只取最後一個值，並徹底轉為 float
if not vix_p.empty:
    # 這裡用 values.flatten() 確保不管它是 Series 還是 DataFrame 都能降維
    latest_vix_p = float(vix_p.values.flatten()[-1])
else:
    latest_vix_p = np.nan

# 3. 定義核心計算函數 (逐檔處理)
def get_indicators(Ticker):
    results = [] # 用來收集所有股票的最新數據
    
    for ticker in Ticker:
        # 1. 明確設定 auto_adjust=False 嘗試保留 Adj Close 欄位
        df = yf.download(ticker, period="2y", auto_adjust=False)
        
        # 確保有足夠的資料列數計算 MA200 (至少 200 天)
        if df.empty or len(df) < 200:
            continue
            
        # 2. 處理 yfinance 新版可能的 MultiIndex 欄位問題
        # 使用 get_level_values(0) 抽取第一層 (即 Price 名稱層) 會比 droplevel 更穩定
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 3. 欄位防呆機制：優先使用 Adj Close，若無則降級使用 Close
        if 'Adj Close' in df.columns:
            df['adj_price'] = df['Adj Close']
        elif 'Close' in df.columns:
            df['adj_price'] = df['Close']
        else:
            print(f"警告：{ticker} 缺少價格欄位，略過此檔。")
            continue
        
        # --- 計算技術指標 ---
        df['ma200'] = ta.sma(df['adj_price'], length=200)
        df['rsi'] = ta.rsi(df['adj_price'], length=14)

        # ADX
        adx_df = ta.adx(df['High'], df['Low'], df['adj_price'], length=14)
        df['adx'] = adx_df['ADX_14'] if adx_df is not None else np.nan

        # ATR 相關
        df['atr'] = ta.atr(df['High'], df['Low'], df['adj_price'], length=14)
        df['atr_60d_avg'] = df['atr'].rolling(window=60).mean()

        # --- 百分位指標 (Percentile) ---
        # BBW Percentile
        bbands = ta.bbands(df['adj_price'], length=20, std=2)
        if bbands is not None and not bbands.empty:
            # 動態比對並取得正確的欄位名稱
            bbu_col = [col for col in bbands.columns if 'BBU' in col][0]
            bbl_col = [col for col in bbands.columns if 'BBL' in col][0]
            bbm_col = [col for col in bbands.columns if 'BBM' in col][0]
            
            # 使用抓取到的正確名稱進行計算
            bbw = (bbands[bbu_col] - bbands[bbl_col]) / bbands[bbm_col]
            df['bbw_percentile'] = bbw.rolling(252).rank(pct=True)
        else:
            df['bbw_percentile'] = np.nan
                
        # HV Percentile
        log_ret = np.log(df['adj_price'] / df['adj_price'].shift(1))
        hv = log_ret.rolling(20).std() * np.sqrt(252)
        df['hv_percentile'] = hv.rolling(252).rank(pct=True)

        # --- 基本面 YoY ---
        tk = yf.Ticker(ticker)
        financials = tk.financials
        
        if financials is not None and 'Total Revenue' in financials.index:
            rev = financials.loc['Total Revenue']
            yoy = rev.pct_change(periods=-1) # 計算年增率
            yoy_now = yoy.iloc[0] if len(yoy) > 0 else np.nan
            yoy_t1 = yoy.iloc[1] if len(yoy) > 1 else np.nan
            yoy_t2 = yoy.iloc[2] if len(yoy) > 2 else np.nan
        else:
            yoy_now = yoy_t1 = yoy_t2 = np.nan

        # --- 整合該檔股票最後一天的所有數據 ---
        latest_data = {
            'ticker': ticker,
            # 使用 float(np.array(...).flatten()[-1]) 是最保險的「脫殼」方式
            'adj_price': round(float(df['adj_price'].values.flatten()[-1]), 2),
            'ma200': round(float(df['ma200'].values.flatten()[-1]), 2),
            'adx': round(float(df['adx'].values.flatten()[-1]), 2),
            'rsi': round(float(df['rsi'].values.flatten()[-1]), 2),
            'atr': round(float(df['atr'].values.flatten()[-1]), 2),
            'atr_60d_avg': round(float(df['atr_60d_avg'].values.flatten()[-1]), 2),
            'bbw_percentile': round(float(df['bbw_percentile'].values.flatten()[-1]), 4),
            'hv_percentile': round(float(df['hv_percentile'].values.flatten()[-1]), 4),
            'vix_percentile': round(latest_vix_p, 4), 
            'yoy_now': round(float(yoy_now), 4) if pd.notna(yoy_now) else np.nan,
            'yoy_t1': round(float(yoy_t1), 4) if pd.notna(yoy_t1) else np.nan,
            'yoy_t2': round(float(yoy_t2), 4) if pd.notna(yoy_t2) else np.nan
        }
        
        # 將字典加入清單
        results.append(latest_data)
            
    # 將包含所有股票最新數據的清單轉為 DataFrame
    final_df = pd.DataFrame(results)
    if not final_df.empty:
        final_df = final_df.set_index('ticker')
        
    return final_df

# 4. 執行並顯示最終結果
final_report = get_indicators(Ticker)

# 如果在 Jupyter Notebook 中，建議使用 display 使排版更易讀
display(final_report)


In [ ]:
def get_single_indicator(self, ticker):
        
        try:
            # 加上 progress=False 避免多執行緒時噴出一堆進度條
            df = yf.download(ticker, period=self.period, auto_adjust=False, progress=False)
            
            if df.empty or len(df) < 200:
                return None

            # --- 重點 1：強力清洗欄位名稱 ---
            # 不管是幾層索引，全部壓平並取最外層，並轉為純字串 + 去除空格
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.columns = [str(c).strip() for c in df.columns]

            # --- 重點 2：更穩健的 adj_price 建立方式 ---
            if 'Adj Close' in df.columns:
                df['adj_price'] = df['Adj Close']
            elif 'Close' in df.columns:
                df['adj_price'] = df['Close']
            else:
                # 在函式裡要用 return None，不能用 continue
                print(f"警告：{ticker} 找不到價格欄位 {df.columns}")
                return None
            
            # 確保 adj_price 裡面沒有 NaN (有時候最新一天會是空的)
            df = df.dropna(subset=['adj_price'])
            
            # --- 計算技術指標 ---
            df['ma200'] = ta.sma(df['adj_price'], length=200)
            df['rsi'] = ta.rsi(df['adj_price'], length=14)

            # ADX
            adx_df = ta.adx(df['High'], df['Low'], df['adj_price'], length=14)
            df['adx'] = adx_df['ADX_14'] if adx_df is not None else np.nan

            # ATR 相關
            df['atr'] = ta.atr(df['High'], df['Low'], df['adj_price'], length=14)
            df['atr_60d_avg'] = df['atr'].rolling(window=60).mean()

            # --- 百分位指標 (Percentile) ---
            # BBW Percentile
            bbands = ta.bbands(df['adj_price'], length=20, std=2)
            if bbands is not None and not bbands.empty:
                # 動態比對並取得正確的欄位名稱
                bbu_col = [col for col in bbands.columns if 'BBU' in col][0]
                bbl_col = [col for col in bbands.columns if 'BBL' in col][0]
                bbm_col = [col for col in bbands.columns if 'BBM' in col][0]
                
                # 使用抓取到的正確名稱進行計算
                bbw = (bbands[bbu_col] - bbands[bbl_col]) / bbands[bbm_col]
                df['bbw_percentile'] = bbw.rolling(252).rank(pct=True)
            else:
                df['bbw_percentile'] = np.nan
                    
            # HV Percentile
            log_ret = np.log(df['adj_price'] / df['adj_price'].shift(1))
            hv = log_ret.rolling(20).std() * np.sqrt(252)
            df['hv_percentile'] = hv.rolling(252).rank(pct=True)

            # --- 基本面 YoY ---
            tk = yf.Ticker(ticker)
            financials = tk.financials
            
            if financials is not None and 'Total Revenue' in financials.index:
                rev = financials.loc['Total Revenue']
                yoy = rev.pct_change(periods=-1) # 計算年增率
                yoy_now = yoy.iloc[0] if len(yoy) > 0 else np.nan
                yoy_t1 = yoy.iloc[1] if len(yoy) > 1 else np.nan
                yoy_t2 = yoy.iloc[2] if len(yoy) > 2 else np.nan
            else:
                yoy_now = yoy_t1 = yoy_t2 = np.nan

            # --- 整合該檔股票最後一天的所有數據 ---
            latest_data = {
                'ticker': ticker,
                # 使用 float(np.array(...).flatten()[-1]) 是最保險的「脫殼」方式
                'adj_price': round(float(df['adj_price'].values.flatten()[-1]), 2),
                'ma200': round(float(df['ma200'].values.flatten()[-1]), 2),
                'adx': round(float(df['adx'].values.flatten()[-1]), 2),
                'rsi': round(float(df['rsi'].values.flatten()[-1]), 2),
                'atr': round(float(df['atr'].values.flatten()[-1]), 2),
                'atr_60d_avg': round(float(df['atr_60d_avg'].values.flatten()[-1]), 2),
                'bbw_percentile': round(float(df['bbw_percentile'].values.flatten()[-1]), 4),
                'hv_percentile': round(float(df['hv_percentile'].values.flatten()[-1]), 4),
                'vix_percentile': round(self.latest_vix_p, 4) if pd.notna(self.latest_vix_p) else np.nan, 
                'yoy_now': round(float(yoy_now), 4) if pd.notna(yoy_now) else np.nan,
                'yoy_t1': round(float(yoy_t1), 4) if pd.notna(yoy_t1) else np.nan,
                'yoy_t2': round(float(yoy_t2), 4) if pd.notna(yoy_t2) else np.nan
            }
            
            return latest_data
        except Exception as e:
            print(f"無法抓取 {ticker}: {e}")
            return None
        

In [ ]:
#2026.3.17修改前
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np

class Indicators:
    def __init__(self, period="2y", length=200):
        self.period = period
        self.length = length

    def get_vix_percentile(self, df):
        # 檢查是否已經計算過 VIX，若無則進行第一次下載與計算
        if not hasattr(self, 'latest_vix_p'):
            vix_raw = yf.download("^VIX", period="2y", progress=False)['Close']
            vix_p = vix_raw.rolling(252).rank(pct=True)
            
            if not vix_p.empty:
                self.latest_vix_p = float(vix_p.values.flatten()[-1])
            else:
                self.latest_vix_p = np.nan

        # 直接套用已儲存（快取）的單一數值到當前 DataFrame
        df['vix_percentile'] = self.latest_vix_p
        return df
    
    def get_adjusted_price(self, df):
        # 1. 處理 yfinance 新版可能回傳 MultiIndex 的狀況
        if isinstance(df.columns, pd.MultiIndex):
            # 取第一層索引 (例如將 ('Adj Close', 'AAPL') 變成 'Adj Close')
            df.columns = df.columns.get_level_values(0)
            
        # 2. 判斷並建立 adj_price 欄位
        if 'Adj Close' in df.columns:
            df['adj_price'] = df['Adj Close']
        elif 'Close' in df.columns:
            df['adj_price'] = df['Close']
        else:
            raise ValueError(f"找不到價格欄位 {df.columns}")
            
        return df

    def get_ma200(self, df):
        df['ma200'] = ta.sma(df['adj_price'], length=self.length) # 使用 init 設定的 length
        return df
    
    def get_rsi(self, df):
        df['rsi'] = ta.rsi(df['adj_price'], length=14)
        return df
    
    def get_adx(self, df):
        adx_df = ta.adx(df['High'], df['Low'], df['adj_price'], length=14)
        if adx_df is not None and not adx_df.empty:
            # 動態抓取 ADX 欄位名稱 (通常為 ADX_14)
            adx_col = [col for col in adx_df.columns if col.startswith('ADX')][0]
            df['adx'] = adx_df[adx_col]
        else:
            df['adx'] = np.nan
        return df

    def get_atr(self, df):
        atr_series = ta.atr(df['High'], df['Low'], df['adj_price'], length=14)
        df['atr'] = atr_series
        return df
    
    def get_atr_60d_avg(self, df):
        if 'atr' in df.columns:
            df['atr_60d_avg'] = df['atr'].rolling(60).mean()
        return df
    
    def get_bbw_percentile(self, df):
        bbands = ta.bbands(df['adj_price'], length=20, std=2)
        if bbands is not None and not bbands.empty:
            bbu_col = [col for col in bbands.columns if 'BBU' in col][0]
            bbl_col = [col for col in bbands.columns if 'BBL' in col][0]
            bbm_col = [col for col in bbands.columns if 'BBM' in col][0]
            
            bbw = (bbands[bbu_col] - bbands[bbl_col]) / bbands[bbm_col]
            df['bbw_percentile'] = bbw.rolling(252).rank(pct=True)
        else:
            df['bbw_percentile'] = np.nan
        return df
    
    def get_hv_percentile(self, df):
        log_returns = np.log(df['adj_price'] / df['adj_price'].shift(1))
        hv = log_returns.rolling(20).std() * np.sqrt(252)
        df['hv_percentile'] = hv.rolling(252).rank(pct=True)
        return df
    
    def get_yoy(self, df, ticker):
        if not ticker:
            df['yoy_now'] = df['yoy_t1'] = df['yoy_t2'] = np.nan
            return df
            
        tk = yf.Ticker(ticker)
        financials = tk.financials
        if financials is not None and 'Total Revenue' in financials.index:
            rev = financials.loc['Total Revenue']
            yoy = rev.pct_change(periods=-1)
            yoy_now = yoy.iloc[0] if len(yoy) > 0 else np.nan
            yoy_t1 = yoy.iloc[1] if len(yoy) > 1 else np.nan
            yoy_t2 = yoy.iloc[2] if len(yoy) > 2 else np.nan
        else:
            yoy_now = yoy_t1 = yoy_t2 = np.nan
            
        df['yoy_now'] = yoy_now
        df['yoy_t1'] = yoy_t1
        df['yoy_t2'] = yoy_t2
        return df
    
    def get_indicators(self, df):
        df = self.get_adjusted_price(df)
        df = self.get_ma200(df)
        df = self.get_rsi(df)
        df = self.get_adx(df)
        df = self.get_atr(df)
        df = self.get_atr_60d_avg(df)
        df = self.get_bbw_percentile(df)
        df = self.get_vix_percentile(df) # 修正：傳入 df 並確保方法名稱一致
        df = self.get_hv_percentile(df)
        
        # 修正：確保有 ticker 才呼叫，避免報錯
        ticker = df['ticker'].iloc[0] if 'ticker' in df.columns else None
        df = self.get_yoy(df, ticker=ticker)
        
        return df

In [ ]:
#2026.3.17修改後
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np

class Indicators:
    def __init__(self, period="2y", length=200):
        self.period = period
        self.length = length
        self.latest_vix_p = None

    def _prepare_price_data(self, df):
        """
        確保 adj_price 存在。
        優先序：現有的 adj_price > close
        """
        # 處理 yfinance 可能產生的 MultiIndex (防禦性程式碼)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0).str.lower()
        else:
            # 統一轉換為小寫以符合你的 Schema
            df.columns = [col.lower() for col in df.columns]

        # 核心邏輯：若無 adj_price 或內容為空，則使用 close
        if 'adj_price' not in df.columns or df['adj_price'].isnull().all():
            if 'close' in df.columns:
                df['adj_price'] = df['close']
            else:
                raise ValueError("資料源缺乏 'adj_price' 與 'close' 欄位，無法計算指標。")
        
        return df

    def get_vix_percentile(self, df):
        # 檢查是否已經計算過 VIX，若無則進行第一次下載與計算
        if self.latest_vix_p is None:
            try:
                vix_raw = yf.download("^VIX", period=self.period, progress=False)['Close']
                # 處理 yf 回傳 Series 或 DataFrame 的情況
                vix_series = vix_raw.iloc[:, 0] if isinstance(vix_raw, pd.DataFrame) else vix_raw
                vix_p = vix_series.rolling(252).rank(pct=True)
                
                if not vix_p.empty:
                    self.latest_vix_p = float(vix_p.iloc[-1])
                else:
                    self.latest_vix_p = np.nan
            except Exception as e:
                print(f"VIX 下載失敗: {e}")
                self.latest_vix_p = np.nan

        df['vix_percentile'] = self.latest_vix_p
        return df

    def get_ma200(self, df):
        df['ma200'] = ta.sma(df['adj_price'], length=self.length)
        return df
    
    def get_rsi(self, df):
        df['rsi'] = ta.rsi(df['adj_price'], length=14)
        return df
    
    def get_adx(self, df):
        # 使用統一的小寫欄位名稱 high, low, adj_price
        adx_df = ta.adx(df['high'], df['low'], df['adj_price'], length=14)
        if adx_df is not None and not adx_df.empty:
            # 抓取第一欄 (通常是 ADX_14)
            df['adx'] = adx_df.iloc[:, 0]
        else:
            df['adx'] = np.nan
        return df

    def get_atr(self, df):
        # 使用統一的小寫欄位名稱 high, low, adj_price
        df['atr'] = ta.atr(df['high'], df['low'], df['adj_price'], length=14)
        return df
    
    def get_atr_60d_avg(self, df):
        if 'atr' in df.columns:
            df['atr_60d_avg'] = df['atr'].rolling(60).mean()
        return df
    
    def get_bbw_percentile(self, df):
        bbands = ta.bbands(df['adj_price'], length=20, std=2)
        if bbands is not None and not bbands.empty:
            # BBW = (Upper - Lower) / Middle
            # bbands 欄位順序通常為 [Lower, Mid, Upper, Bandwidth, %B]
            # 我們直接取 Bandwidth (通常是第 4 欄，名稱含 BBB)
            bbw_col = [col for col in bbands.columns if 'BBB' in col]
            if bbw_col:
                bbw = bbands[bbw_col[0]]
            else:
                # 萬一名稱不對，手動計算
                bbw = (bbands.iloc[:, 2] - bbands.iloc[:, 0]) / bbands.iloc[:, 1]
            
            df['bbw_percentile'] = bbw.rolling(252).rank(pct=True)
        else:
            df['bbw_percentile'] = np.nan
        return df
    
    def get_hv_percentile(self, df):
        # 歷史波動率：對數收益率的標準差
        log_returns = np.log(df['adj_price'] / df['adj_price'].shift(1))
        hv = log_returns.rolling(20).std() * np.sqrt(252)
        df['hv_percentile'] = hv.rolling(252).rank(pct=True)
        return df
    
    def get_yoy(self, df, ticker):
        if not ticker:
            df['yoy_now'] = df['yoy_t1'] = df['yoy_t2'] = np.nan
            return df
            
        try:
            tk = yf.Ticker(ticker)
            financials = tk.financials
            if financials is not None and not financials.empty and 'Total Revenue' in financials.index:
                rev = financials.loc['Total Revenue']
                yoy = rev.pct_change(periods=-1) # yfinance 財報通常是由新到舊排
                df['yoy_now'] = yoy.iloc[0] if len(yoy) > 0 else np.nan
                df['yoy_t1'] = yoy.iloc[1] if len(yoy) > 1 else np.nan
                df['yoy_t2'] = yoy.iloc[2] if len(yoy) > 2 else np.nan
            else:
                df['yoy_now'] = df['yoy_t1'] = df['yoy_t2'] = np.nan
        except:
            df['yoy_now'] = df['yoy_t1'] = df['yoy_t2'] = np.nan
        return df
    
    def get_indicators(self, df):
        # 1. 預處理：對齊 Schema 並處理 adj_price 回退邏輯
        df = self._prepare_price_data(df)
        
        # 2. 計算基礎技術指標
        df = self.get_ma200(df)
        df = self.get_rsi(df)
        df = self.get_adx(df)
        df = self.get_atr(df)
        df = self.get_atr_60d_avg(df)
        
        # 3. 計算統計類指標 (Percentile)
        df = self.get_bbw_percentile(df)
        df = self.get_hv_percentile(df)
        df = self.get_vix_percentile(df)
        
        # 4. 基本面指標 (YoY)
        # 嘗試從 dataframe 抓取 ticker，若無則傳 None
        ticker_val = df['ticker'].iloc[0] if 'ticker' in df.columns else None
        df = self.get_yoy(df, ticker=ticker_val)
        
        return df

In [ ]:
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
from indicators import Indicators
import yfinance_fetcher

ticker = "1301.TW, 1101.TW"
ind=Indicators()
fetcher = yfinance_fetcher.Yfinance_fetcher()
df = fetcher.fetch(ticker, period="2y")
df = ind.get_indicators(df)
display(df)

In [ ]:
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
from indicators import Indicators
import yfinance_fetcher

ticker = "1301.TW, 1101.TW"
ind=Indicators()
fetcher = yfinance_fetcher.Yfinance_fetcher()
df = fetcher.fetch(ticker, period="2y")
df = ind.get_yoy(df)
display(df)